In [ ]:
"""
Example: retrieve New England hourly LCD for summer (JJA) 2015, then derive
precipitation-event durations and a within-day lag. Timestamps are UTC.
"""

import sys
from pathlib import Path

sys.path.insert(
    0,
    "/users/jkodero/research/local_climatological_data",
)


import lcd


In [ ]:
NEW_ENG_LAT_MAX = 47.5
NEW_ENG_LAT_MIN = 40.0
NEW_ENG_LON_MAX = -66.5
NEW_ENG_LON_MIN = -74.0

YEAR = 2015
JJA = [6, 7, 8]
OUTPUT = Path("station.ne.2015.jja.nc")


# Download, clean, classify, and store as netCDF. Returns the .nc path.
path = lcd.get_lcd_from_noaa(
    lon_min=NEW_ENG_LON_MIN,
    lon_max=NEW_ENG_LON_MAX,
    lat_min=NEW_ENG_LAT_MIN,
    lat_max=NEW_ENG_LAT_MAX,
    min_year=YEAR,
    max_year=YEAR,
    months=JJA,
    classify_convective=True,
    as_netcdf=True,
    output=OUTPUT,
    workers=8,
)
print(f"wrote {path}")


2026-07-21 20:07:35,820  INFO  Selected 134 stations
2026-07-21 20:07:35,821  INFO  Downloading 134 station-year files
2026-07-21 20:07:51,297  INFO  Download tally: {'ok': 125, 'missing': 9, 'error': 0}
2026-07-21 20:07:51,310  INFO  Cleaning 125 station files


TypeError: Cannot interpret '<StringDtype(na_value=nan)>' as a data type

In [ ]:
# Load as a long DataFrame (UTC), and as an xarray Dataset (station, time).
df = lcd.open_dataset(path, engine="pandas")
ds = lcd.open_dataset(path, engine="netcdf")
print(
    f"records: {len(df):,}   stations: {ds.sizes['station']}   "
    f"hours: {ds.sizes['time']}"
)
print(df["prec_type"].value_counts().to_dict())

# Local Standard Time copy for local-hour analysis (original stays UTC).
local = lcd.to_local_time(df)
print("UTC range   :", df["time"].min(), "->", df["time"].max())
print("local range :", local["time"].min(), "->", local["time"].max())

# Precipitation-event durations (minutes) and a 1-hour within-day lag,
# each saved to its own netCDF.
lcd.get_durations(df, output="durations.ne.2015.jja.nc")
lcd.get_lag(df, lag=1, output="lag1.ne.2015.jja.nc")

# Convective wet hours only, via the retained au/aw/mw classification.
convective = df[df["prec_type"] == "convective"]
print(
    f"convective wet hours: {len(convective):,}  "
    f"mean intensity: {convective['prec'].mean():.3f} mm/hr"
)
